# CNN Transfer — Mechatronics Vision 2026

Three sections. Earlier sections are the recommended path for the current
data size (~40 labeled images). Later sections are research-oriented and
gated on having much more unlabeled data.

**Section A — YOLO11 fine-tune from COCO weights (primary path).**
Strongest baseline at current scale. Uses the annotation-aware augmentation
pipeline defined in `augmentation_pipeline.ipynb`.

**Section B — Supervised Contrastive auxiliary loss (optional).**
Adds a projection head on the YOLO backbone. Pulls same-class instance
embeddings together, pushes different-class apart. Runs alongside the
detection loss. Uses the true labels — does **not** discover labels.

**Section C — Self-supervised DINO + DBSCAN vs ground truth (gated).**
Runs only if you have ≥ `GATE_MIN_UNLABELED` raw frames. Trains a
self-distillation model, clusters instance embeddings with DBSCAN + k-means,
and compares cluster assignments to true labels via ARI, NMI, and
Hungarian-matched accuracy. The point of this section is not to replace
Section A but to measure whether self-supervised learning would buy you
anything at your data scale before investing real training time in it.

**Rationale for this structure.** At ~40 labeled images, self-supervised
contrastive pre-training underperforms a COCO-pretrained backbone fine-tuned
directly — the SSL signal is too weak with so few views. Once the ZedBox has
captured hundreds-to-thousands of unlabeled raw frames, Section C becomes
meaningful. Section B is the only "contrastive" variant worth running at the
current scale.

In [ ]:
import os, sys, json, math, random, shutil, copy
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import yaml
import cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
import matplotlib.pyplot as plt

DEVICE = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
print('device:', DEVICE, '| torch', torch.__version__)

## 0. Configuration (shared)

Imported from `vision_config.py` — the same file `augmentation_pipeline.ipynb`
and `deploy.py` read. Flip `LABEL_KIND` in that file to toggle between
polygon-segmentation training and bounding-box detection training; every
path and model choice below follows automatically.

**Note on `yolo_dataset/`.** Section A.1 builds an Ultralytics-style
`images/{train,val}` + `labels/{train,val}` folder there **using symlinks**
back into `data/`. Your real images and labels stay in `data/` —
`yolo_dataset/` is a rebuildable artifact that Ultralytics consumes.

In [ ]:
import sys
sys.path.insert(0, '/Users/jnoael/Mechatronics_Vision_2026')
from vision_config import *   # all shared settings

# Auto-resolved from LABEL_KIND. Override in this cell for one-off experiments.
LABELS_DIR = resolved_labels_dir()
FT_MODEL   = resolved_yolo_model()

RUNS_DIR.mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
summary()
print(f'  active labels dir: {LABELS_DIR}')
print(f'  fine-tune base   : {FT_MODEL}')
print(f'  runs dir         : {RUNS_DIR}')


## 0b. Re-import key helpers from the augmentation notebook

Self-contained copies so this notebook runs standalone. Keep in sync with
`augmentation_pipeline.ipynb` (Sections 2–5).

In [ ]:
def detect_format(tokens):
    n = len(tokens)
    if n == 5: return 'bbox'
    if n >= 7 and (n - 1) % 2 == 0: return 'polygon'
    raise ValueError(f'unrecognized label shape: {n} tokens')

def read_label(path):
    items = []
    text = Path(path).read_text().strip()
    if not text: return items
    for line in text.splitlines():
        parts = line.split()
        if not parts: continue
        cls  = int(parts[0])
        nums = [float(x) for x in parts[1:]]
        kind = detect_format([cls] + nums)
        if kind == 'bbox':
            items.append(dict(cls=cls, kind='bbox',
                              coords=np.array(nums, np.float32)))
        else:
            pts = np.array(nums, np.float32).reshape(-1, 2)
            items.append(dict(cls=cls, kind='polygon', coords=pts))
    return items

def poly_to_bbox(pts):
    x_min, y_min = pts.min(axis=0); x_max, y_max = pts.max(axis=0)
    return np.array([(x_min+x_max)/2, (y_min+y_max)/2,
                     x_max-x_min,    y_max-y_min], np.float32)

def polys_to_mask_stack(polys, h, w):
    stack = np.zeros((len(polys), h, w), np.uint8)
    for i, poly in enumerate(polys):
        pix = (poly * np.array([w, h])).round().astype(np.int32)
        cv2.fillPoly(stack[i], [pix], 1)
    return stack

def crop_instance(img, poly, pad_frac=0.10, out_size=224):
    h, w = img.shape[:2]
    x_min, y_min = poly.min(axis=0); x_max, y_max = poly.max(axis=0)
    px = (x_max - x_min) * pad_frac; py = (y_max - y_min) * pad_frac
    x0 = int(max(0, (x_min - px) * w)); y0 = int(max(0, (y_min - py) * h))
    x1 = int(min(w, (x_max + px) * w)); y1 = int(min(h, (y_max + py) * h))
    if x1 <= x0 or y1 <= y0: return None
    return cv2.resize(img[y0:y1, x0:x1], (out_size, out_size))

def build_aug(img_size=IMAGE_SIZE, train=True):
    """Condensed aug pipeline driven by shared AUG_PROBS from vision_config.
    Full pipeline lives in augmentation_pipeline.ipynb."""
    p = AUG_PROBS
    geo = []
    if train:
        geo = [
            A.HorizontalFlip(p=p['h_flip']),
            A.VerticalFlip(p=p['v_flip']),
            A.RandomRotate90(p=p['rotate_90']),
            A.Affine(rotate=(-ROTATE_DEG, ROTATE_DEG),
                     shear={'x': (-SKEW_DEG, SKEW_DEG),
                            'y': (-SKEW_DEG, SKEW_DEG)},
                     p=max(p['rotate_small'], p['affine_skew'])),
            A.RandomResizedCrop(size=(img_size, img_size),
                                scale=CROP_SCALE, ratio=(0.85, 1.17),
                                p=p['random_crop']),
        ]
    geo.append(A.Resize(img_size, img_size))
    photo = []
    if train:
        photo = [
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3, hue=0.1,
                          p=p['color_jitter']),
            A.ChannelDropout(p=p['channel_dropout']),
            A.CoarseDropout(num_holes_range=CUTOUT_HOLES,
                            hole_height_range=CUTOUT_FRAC,
                            hole_width_range=CUTOUT_FRAC,
                            fill=0, p=p['cutout']),
            A.GaussNoise(std_range=(0.04, 0.12), p=p['gauss_noise']),
            A.GaussianBlur(blur_limit=(3, 7), p=p['gauss_blur']),
        ]
    return A.Compose(geo + photo)


## A. YOLO11 Fine-Tune from COCO (primary path)

### A.1 Train/val split

Classes are identified by filename prefix. We split each class 80/20 so both
folds see every class.

In [ ]:
def build_split(images_dir: Path, labels_dir: Path,
                 out_dir: Path, val_fraction: float = VAL_FRACTION,
                 seed: int = SEED) -> Path:
    """Materialize an Ultralytics-style dataset folder with symlinks and a
    YOLO data.yaml. Stratifies by filename prefix."""
    rng = random.Random(seed)
    by_prefix = {}
    for img in sorted(images_dir.iterdir()):
        if img.suffix.lower() not in {'.png', '.jpg', '.jpeg'}: continue
        prefix = img.stem.split('_')[0]
        by_prefix.setdefault(prefix, []).append(img)

    out_dir.mkdir(parents=True, exist_ok=True)
    for split in ('train', 'val'):
        (out_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
        (out_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

    for prefix, imgs in by_prefix.items():
        rng.shuffle(imgs)
        k = max(1, int(round(len(imgs) * val_fraction)))
        val_set = set(imgs[:k]); train_set = set(imgs[k:])
        for img in imgs:
            split = 'val' if img in val_set else 'train'
            lab = labels_dir / (img.stem + '.txt')
            if not lab.exists(): continue
            dst_i = out_dir / 'images' / split / img.name
            dst_l = out_dir / 'labels' / split / lab.name
            if dst_i.exists(): dst_i.unlink()
            if dst_l.exists(): dst_l.unlink()
            dst_i.symlink_to(img); dst_l.symlink_to(lab)

    yaml_path = out_dir / 'data.yaml'
    yaml_path.write_text(yaml.safe_dump({
        'path':  str(out_dir),
        'train': 'images/train',
        'val':   'images/val',
        'names': CLASS_NAMES,
    }, sort_keys=False))
    print('wrote', yaml_path)
    return yaml_path

# yolo_dataset/ is a symlink-only build folder. Real data stays in data/.
DATA_YAML = build_split(IMAGES_DIR, LABELS_DIR, YOLO_DATASET_DIR)
print(Path(DATA_YAML).read_text())


### A.2 Fine-tune

Ultralytics `YOLO.train` has built-in augmentation (mosaic, mixup, hsv,
flips). We enable those and let our external `augmentation_pipeline.ipynb`
cover the additional modes (Sobel, channel dropout, etc.) via the
optional bulk export. If you want *only* our pipeline, export augmented
copies with `bulk_export(...)` in the aug notebook and point `DATA_YAML`
at that folder.

In [ ]:
def finetune_yolo(data_yaml: Path, model: str = FT_MODEL,
                   epochs: int = FT_EPOCHS, batch: int = FT_BATCH,
                   imgsz: int = IMAGE_SIZE, project: Path = RUNS_DIR):
    from ultralytics import YOLO
    m = YOLO(model)
    results = m.train(
        data=str(data_yaml), epochs=epochs, batch=batch, imgsz=imgsz,
        project=str(project), name='ft', exist_ok=True,
        device=0 if DEVICE == 'cuda' else DEVICE,
        # Ultralytics built-in aug (tuned up)
        hsv_h=0.02, hsv_s=0.7, hsv_v=0.5,
        degrees=15.0, translate=0.1, scale=0.5, shear=5.0, perspective=0.0,
        flipud=0.1, fliplr=0.5,
        mosaic=1.0, mixup=0.1, copy_paste=0.1,
        erasing=0.2,
    )
    return m, results

# ▶ Actually train. Weights land in runs_cnn_transfer/ft/weights/best.pt
model, results = finetune_yolo(DATA_YAML)
print('best weights:', model.trainer.best)


### A.3 Evaluate

Produces mAP, per-class metrics, and writes confusion matrix + PR curves to
`runs_cnn_transfer/ft/`.

In [ ]:
def evaluate_yolo(weights_path: Path, data_yaml: Path, imgsz: int = IMAGE_SIZE):
    from ultralytics import YOLO
    m = YOLO(str(weights_path))
    metrics = m.val(data=str(data_yaml), imgsz=imgsz, plots=True,
                    device=0 if DEVICE == 'cuda' else DEVICE)
    print('mAP50:',    metrics.box.map50)
    print('mAP50-95:', metrics.box.map)
    return metrics

best_weights = RUNS_DIR / 'ft' / 'weights' / 'best.pt'
if best_weights.exists():
    _ = evaluate_yolo(best_weights, DATA_YAML)
else:
    print('no weights yet — run the training cell above first')

### A.4 Resume / Further training

Two small helpers for the two common post-run paths:

- **`resume_training`** — a run crashed mid-epoch. Ultralytics saves
  `last.pt` + `args.yaml` after each epoch in the run folder; this helper
  loads them and picks up at the next epoch with identical config.
- **`continue_training`** — a run finished cleanly and you want to train
  the resulting `best.pt` for N more epochs on new/expanded data (e.g.
  after regenerating Blender renders). Starts a *fresh* run that
  warm-starts from those weights, writes to a different `run_name` so
  the original run stays intact.


In [ ]:
def resume_training(run_name: str = 'ft', project: Path = RUNS_DIR):
    """Resume an interrupted run from its last.pt. Reads the run's
    args.yaml automatically and continues at the next epoch."""
    from ultralytics import YOLO
    last_pt = project / run_name / 'weights' / 'last.pt'
    if not last_pt.exists():
        raise FileNotFoundError(
            f'No last.pt at {last_pt}. Either no run exists yet, or it '
            f'finished cleanly (only best.pt remains). Use '
            f'continue_training() to further-train a finished best.pt.'
        )
    print(f'resuming from {last_pt}')
    m = YOLO(str(last_pt))
    results = m.train(resume=True)
    return m, results


def continue_training(weights: Path, data_yaml: Path,
                      epochs: int = 30, batch: int = FT_BATCH,
                      run_name: str = 'ft_continue',
                      project: Path = RUNS_DIR):
    """Warm-start a fresh run from an existing best.pt. Use after
    generating new data to train for a few dozen more epochs without
    touching the original run folder."""
    from ultralytics import YOLO
    weights = Path(weights)
    if not weights.exists():
        raise FileNotFoundError(f'weights not found: {weights}')
    print(f'continuing from {weights} -> {project / run_name}')
    m = YOLO(str(weights))
    results = m.train(
        data=str(data_yaml), epochs=epochs, batch=batch,
        imgsz=IMAGE_SIZE, project=str(project), name=run_name, exist_ok=True,
        device=0 if DEVICE == 'cuda' else DEVICE,
        # Same aug knobs as the initial finetune (cell A.2) for consistency
        hsv_h=0.02, hsv_s=0.7, hsv_v=0.5,
        degrees=15.0, translate=0.1, scale=0.5, shear=5.0, perspective=0.0,
        flipud=0.1, fliplr=0.5,
        mosaic=1.0, mixup=0.1, copy_paste=0.1,
        erasing=0.2,
    )
    return m, results

# -- Usage (uncomment when you need it) -----------------------------

# Crash recovery on the default 'ft' run:
# resume_training(run_name='ft')

# Warm-start fine-tune on freshly regenerated data. Rebuild the split
# first so the new images/labels are picked up:
# DATA_YAML = build_split(IMAGES_DIR, LABELS_DIR, YOLO_DATASET_DIR)
# continue_training(
#     weights   = RUNS_DIR / 'ft' / 'weights' / 'best.pt',
#     data_yaml = DATA_YAML,
#     epochs    = 35,
#     run_name  = 'ft_glare_v1',
# )


## B. Supervised Contrastive Auxiliary (optional)

Adds a projection head on top of the YOLO11 backbone and trains it with
SupCon loss on instance crops. After training, swap the SupCon-tuned
backbone into a fresh YOLO and fine-tune the detection head on top.

**When to use.** If Section A plateaus and you suspect the backbone's
feature space doesn't separate your classes well. Measure via a t-SNE of
embeddings from the plain YOLO backbone first; if clusters are fuzzy,
SupCon is likely to help.

In [ ]:
class InstanceCropDataset(Dataset):
    """Crops each labeled instance and returns (two augmented views, class_id).
    Two views → SupCon positive pair."""
    def __init__(self, images_dir, labels_dir, transform, out_size=224):
        self.entries = []
        for img_p in sorted(Path(images_dir).iterdir()):
            if img_p.suffix.lower() not in {'.png', '.jpg', '.jpeg'}: continue
            lab_p = Path(labels_dir) / (img_p.stem + '.txt')
            if not lab_p.exists(): continue
            for a in read_label(lab_p):
                if a['kind'] == 'polygon':
                    self.entries.append((img_p, a['cls'], a['coords']))
                else:
                    cx, cy, bw, bh = a['coords']
                    x0, y0 = cx-bw/2, cy-bh/2; x1, y1 = cx+bw/2, cy+bh/2
                    rect = np.array([[x0,y0],[x1,y0],[x1,y1],[x0,y1]], np.float32)
                    self.entries.append((img_p, a['cls'], rect))
        self.transform = transform
        self.out_size  = out_size

    def __len__(self): return len(self.entries)

    def _view(self, img, poly):
        crop = crop_instance(img, poly, pad_frac=0.10, out_size=self.out_size)
        if crop is None:
            crop = np.zeros((self.out_size, self.out_size, 3), np.uint8)
        return self.transform(image=crop)['image']

    def __getitem__(self, i):
        img_p, cls, poly = self.entries[i]
        img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
        v1 = self._view(img, poly); v2 = self._view(img, poly)
        to_t = lambda x: torch.from_numpy(x).permute(2,0,1).float() / 255.0
        return to_t(v1), to_t(v2), cls

def supcon_loss(features: torch.Tensor, labels: torch.Tensor,
                temperature: float = SUPCON_TEMP) -> torch.Tensor:
    """SupCon (Khosla et al. 2020). features: (2B, D) L2-normalized.
    labels: (2B,) with matching positions between views."""
    device = features.device
    B2 = features.shape[0]
    labels = labels.view(-1, 1)
    mask = torch.eq(labels, labels.T).float().to(device)
    logits = torch.matmul(features, features.T) / temperature
    logits = logits - logits.max(dim=1, keepdim=True).values.detach()
    eye = torch.eye(B2, device=device)
    mask = mask * (1 - eye)
    exp_logits = torch.exp(logits) * (1 - eye)
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-12)
    mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-12)
    return -mean_log_prob_pos.mean()


In [ ]:
def load_yolo_backbone(weights: str = FT_MODEL):
    """Return YOLO11 backbone as an nn.Module that maps (B,3,H,W) → (B,C,H',W')."""
    from ultralytics import YOLO
    y = YOLO(weights)
    # Ultralytics DetectionModel stores layers in .model.model (a Sequential).
    # Backbone is layers before the neck/head. YOLO11n backbone ends at index 10.
    full = y.model.model
    backbone = nn.Sequential(*list(full.children())[:11])
    return backbone, y

class SupConModel(nn.Module):
    def __init__(self, backbone, proj_dim=SUPCON_PROJ_DIM):
        super().__init__()
        self.backbone = backbone
        # Infer feature dim via a dry run
        with torch.no_grad():
            feat = backbone(torch.zeros(1, 3, 224, 224))
        feat_dim = feat.shape[1]
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(
            nn.Linear(feat_dim, feat_dim), nn.ReLU(inplace=True),
            nn.Linear(feat_dim, proj_dim),
        )
    def forward(self, x):
        f = self.backbone(x)
        f = self.pool(f).flatten(1)
        z = F.normalize(self.proj(f), dim=1)
        return z

def train_supcon(epochs=SUPCON_EPOCHS, batch=SUPCON_BATCH):
    ds = InstanceCropDataset(IMAGES_DIR, LABELS_DIR, build_aug(train=True))
    if len(ds) < batch:
        print(f'[supcon] only {len(ds)} instances — skipping (need ≥ {batch})')
        return None
    dl = DataLoader(ds, batch_size=batch, shuffle=True, drop_last=True, num_workers=2)
    backbone, _ = load_yolo_backbone()
    model = SupConModel(backbone).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

    model.train()
    for ep in range(epochs):
        total = 0.0; n = 0
        for v1, v2, y in dl:
            v1, v2, y = v1.to(DEVICE), v2.to(DEVICE), y.to(DEVICE)
            z = model(torch.cat([v1, v2], dim=0))
            labels_cat = torch.cat([y, y], dim=0)
            loss = supcon_loss(z, labels_cat)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * v1.size(0); n += v1.size(0)
        print(f'epoch {ep+1:03d}  loss={total/max(n,1):.4f}')

    ckpt = RUNS_DIR / 'supcon_backbone.pt'
    torch.save(model.backbone.state_dict(), ckpt)
    print('saved', ckpt)
    return ckpt

# ckpt = train_supcon()


### B.1 Transfer SupCon backbone → YOLO head

Load the SupCon-trained backbone weights into a fresh YOLO, freeze the
backbone for a warm-up phase, then unfreeze everything for full fine-tune.

In [ ]:
def finetune_with_supcon_init(data_yaml: Path, supcon_ckpt: Path,
                                 epochs=FT_EPOCHS, batch=FT_BATCH):
    from ultralytics import YOLO
    m = YOLO(FT_MODEL)
    # Load trained backbone weights into the matching layers
    full = m.model.model
    state = torch.load(supcon_ckpt, map_location='cpu')
    for i, layer in enumerate(list(full.children())[:11]):
        prefix = f'{i}.'
        layer_state = {k[len(prefix):]: v for k, v in state.items()
                       if k.startswith(prefix)}
        if layer_state:
            layer.load_state_dict(layer_state, strict=False)
    print('backbone initialised from', supcon_ckpt)

    results = m.train(data=str(data_yaml), epochs=epochs, batch=batch,
                      imgsz=IMAGE_SIZE, project=str(RUNS_DIR),
                      name='ft_supcon', exist_ok=True,
                      device=0 if DEVICE == 'cuda' else DEVICE,
                      freeze=10)   # freeze backbone for stability in first phase
    return m, results

# ckpt = RUNS_DIR / 'supcon_backbone.pt'
# if ckpt.exists():
#     finetune_with_supcon_init(DATA_YAML, ckpt)

## C. Self-Supervised DINO + Truth Comparison (gated)

Runs a small DINO-style self-distillation on raw (unlabeled) frames, then
extracts embeddings for every labeled instance, clusters with DBSCAN +
k-means, and measures clustering quality against true labels.

**Why gated.** DINO needs lots of unlabeled data to produce useful
invariances. With < 500 raw frames, expect near-chance clustering and
no improvement from transferring DINO weights into YOLO.

Metrics reported:
- **Adjusted Rand Index (ARI)** — 0 = random, 1 = perfect. Invariant to
  label permutation.
- **Normalized Mutual Information (NMI)** — information-theoretic analog.
- **Hungarian-matched accuracy** — optimally aligns cluster IDs to true
  class IDs, then reports classification accuracy on that alignment.
- **Confusion matrix** — visual inspection after Hungarian alignment.

In [ ]:
def count_unlabeled(root: Path) -> int:
    return sum(1 for p in root.rglob('*')
               if p.suffix.lower() in {'.png', '.jpg', '.jpeg'})

def gate_check():
    n = count_unlabeled(IMAGES_DIR)
    print(f'unlabeled image count under {IMAGES_DIR}: {n}')
    print(f'gate threshold: {GATE_MIN_UNLABELED}')
    if n < GATE_MIN_UNLABELED:
        print('→ Section C disabled. Capture more raw frames first.')
        return False
    print('→ Section C enabled.')
    return True

DINO_ENABLED = gate_check()


In [ ]:
class DINOTwoView(Dataset):
    """Two independent augmented views of the same raw frame."""
    def __init__(self, images_dir, transform, out_size=224):
        self.paths = [p for p in sorted(Path(images_dir).rglob('*'))
                      if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}]
        self.transform = transform
        self.out_size  = out_size
    def __len__(self): return len(self.paths)
    def _aug(self, img):
        return self.transform(image=cv2.resize(img, (self.out_size, self.out_size)))['image']
    def __getitem__(self, i):
        img = cv2.cvtColor(cv2.imread(str(self.paths[i])), cv2.COLOR_BGR2RGB)
        v1 = self._aug(img); v2 = self._aug(img)
        to_t = lambda x: torch.from_numpy(x).permute(2,0,1).float() / 255.0
        return to_t(v1), to_t(v2)

class DINOHead(nn.Module):
    def __init__(self, in_dim, out_dim=4096, hidden=2048, bottleneck=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, bottleneck),
        )
        self.last = nn.utils.weight_norm(nn.Linear(bottleneck, out_dim, bias=False))
        self.last.weight_g.data.fill_(1); self.last.weight_g.requires_grad = False
    def forward(self, x):
        x = F.normalize(self.mlp(x), dim=-1, p=2)
        return self.last(x)

class DINOWrap(nn.Module):
    def __init__(self, backbone, in_dim):
        super().__init__()
        self.backbone = backbone
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = DINOHead(in_dim)
    def forward(self, x):
        f = self.pool(self.backbone(x)).flatten(1)
        return self.head(f)

def dino_loss(s_logits, t_logits, center, t_temp=DINO_TEMP_TEACHER,
              s_temp=DINO_TEMP_STUDENT):
    t = F.softmax((t_logits - center) / t_temp, dim=-1).detach()
    s = F.log_softmax(s_logits / s_temp, dim=-1)
    return -(t * s).sum(dim=-1).mean()

@torch.no_grad()
def update_ema(student, teacher, m=DINO_EMA):
    for ps, pt in zip(student.parameters(), teacher.parameters()):
        pt.data.mul_(m).add_(ps.data, alpha=1 - m)

def train_dino(epochs=DINO_EPOCHS, batch=DINO_BATCH):
    if not DINO_ENABLED:
        print('DINO gated off.'); return None
    ds = DINOTwoView(IMAGES_DIR, build_aug(train=True))
    dl = DataLoader(ds, batch_size=batch, shuffle=True, drop_last=True, num_workers=2)
    backbone, _ = load_yolo_backbone()
    with torch.no_grad():
        in_dim = backbone(torch.zeros(1,3,224,224)).shape[1]
    student = DINOWrap(backbone, in_dim).to(DEVICE)
    teacher = copy.deepcopy(student).to(DEVICE)
    for p in teacher.parameters(): p.requires_grad = False
    opt = torch.optim.AdamW(student.parameters(), lr=5e-4, weight_decay=1e-4)
    center = torch.zeros(1, 4096, device=DEVICE)

    student.train(); teacher.train()
    for ep in range(epochs):
        total = 0.0; n = 0
        for v1, v2 in dl:
            v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
            s_out = torch.cat([student(v1), student(v2)], 0)
            with torch.no_grad():
                t_out = torch.cat([teacher(v1), teacher(v2)], 0)
            loss = 0.5 * (dino_loss(s_out[:len(v1)], t_out[len(v1):], center) +
                          dino_loss(s_out[len(v1):], t_out[:len(v1)], center))
            opt.zero_grad(); loss.backward(); opt.step()
            update_ema(student, teacher)
            with torch.no_grad():
                center = 0.9 * center + 0.1 * t_out.mean(dim=0, keepdim=True)
            total += loss.item() * v1.size(0); n += v1.size(0)
        print(f'epoch {ep+1:03d}  loss={total/max(n,1):.4f}')

    ckpt = RUNS_DIR / 'dino_backbone.pt'
    torch.save(student.backbone.state_dict(), ckpt)
    print('saved', ckpt)
    return ckpt

# dino_ckpt = train_dino()


### C.1 Embed instances, cluster, compare to truth

Uses whatever backbone checkpoint is available (DINO > SupCon > plain COCO
YOLO). Extracts one embedding per labeled instance, then runs both DBSCAN
and k-means with `k = NUM_CLASSES`. Reports ARI, NMI, and Hungarian
accuracy for each.

In [ ]:
@torch.no_grad()
def extract_instance_embeddings(backbone_ckpt: Optional[Path] = None,
                                out_size: int = 224) -> Tuple[np.ndarray, np.ndarray]:
    backbone, _ = load_yolo_backbone()
    if backbone_ckpt is not None and Path(backbone_ckpt).exists():
        backbone.load_state_dict(torch.load(backbone_ckpt, map_location='cpu'),
                                 strict=False)
        print('loaded backbone from', backbone_ckpt)
    backbone = backbone.to(DEVICE).eval()
    pool = nn.AdaptiveAvgPool2d(1)

    feats, ys = [], []
    for img_p in sorted(IMAGES_DIR.iterdir()):
        if img_p.suffix.lower() not in {'.png', '.jpg', '.jpeg'}: continue
        lab_p = LABELS_DIR / (img_p.stem + '.txt')
        if not lab_p.exists(): continue
        img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
        for a in read_label(lab_p):
            if a['kind'] == 'polygon': poly = a['coords']
            else:
                cx, cy, bw, bh = a['coords']
                x0, y0 = cx-bw/2, cy-bh/2; x1, y1 = cx+bw/2, cy+bh/2
                poly = np.array([[x0,y0],[x1,y0],[x1,y1],[x0,y1]], np.float32)
            crop = crop_instance(img, poly, pad_frac=0.10, out_size=out_size)
            if crop is None: continue
            t = torch.from_numpy(crop).permute(2,0,1).unsqueeze(0).float().to(DEVICE) / 255.0
            emb = pool(backbone(t)).flatten(1).cpu().numpy()[0]
            feats.append(emb); ys.append(a['cls'])
    return np.stack(feats), np.array(ys, dtype=np.int64)

def compare_clusters_to_truth(y_true: np.ndarray, y_pred: np.ndarray,
                               tag: str = ''):
    from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                                  confusion_matrix)
    from scipy.optimize import linear_sum_assignment

    ari = adjusted_rand_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)

    # Hungarian alignment: only valid for non-noise clusters (id >= 0)
    mask = y_pred >= 0
    aligned = np.full_like(y_pred, fill_value=-1)
    if mask.any():
        cm = confusion_matrix(y_true[mask], y_pred[mask])
        # pad to square
        n = max(cm.shape)
        cm_sq = np.zeros((n, n), dtype=cm.dtype); cm_sq[:cm.shape[0], :cm.shape[1]] = cm
        row, col = linear_sum_assignment(-cm_sq)
        mapping = {c: r for r, c in zip(row, col)}
        aligned[mask] = np.array([mapping.get(int(p), -1) for p in y_pred[mask]])
        acc = (aligned[mask] == y_true[mask]).mean()
    else:
        acc = 0.0

    print(f'[{tag}] ARI={ari:.3f}  NMI={nmi:.3f}  hungarian_acc={acc:.3f}  '
          f'noise={(~mask).sum()}/{len(y_pred)}')
    return dict(ari=ari, nmi=nmi, acc=acc, aligned=aligned, y_true=y_true)

def plot_confusion(aligned: np.ndarray, y_true: np.ndarray, tag: str):
    from sklearn.metrics import confusion_matrix
    mask = aligned >= 0
    cm = confusion_matrix(y_true[mask], aligned[mask],
                          labels=list(CLASS_NAMES.keys()))
    fig, ax = plt.subplots(figsize=(4, 3.5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(list(CLASS_NAMES.values()), rotation=45, ha='right')
    ax.set_yticklabels(list(CLASS_NAMES.values()))
    ax.set_xlabel('predicted (Hungarian-aligned)'); ax.set_ylabel('truth')
    ax.set_title(f'{tag} confusion'); fig.colorbar(im, ax=ax)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black')
    plt.tight_layout(); plt.show()


In [ ]:
def run_cluster_eval(backbone_ckpt: Optional[Path] = None, tag: str = 'plain'):
    from sklearn.cluster import KMeans, DBSCAN
    from sklearn.preprocessing import StandardScaler

    feats, y_true = extract_instance_embeddings(backbone_ckpt)
    print(f'extracted {feats.shape[0]} embeddings, dim={feats.shape[1]}')
    if feats.shape[0] < 2:
        print('not enough instances to cluster'); return

    X = StandardScaler().fit_transform(feats)

    km = KMeans(n_clusters=NUM_CLASSES, n_init=10, random_state=SEED).fit(X)
    res_km = compare_clusters_to_truth(y_true, km.labels_, tag=f'{tag}/kmeans')
    plot_confusion(res_km['aligned'], y_true, tag=f'{tag}/kmeans')

    # DBSCAN eps is dataset-dependent — sweep a small grid and pick best NMI
    from sklearn.metrics import normalized_mutual_info_score
    best = None
    for eps in np.linspace(0.3, 3.0, 10):
        for ms in (2, 3, 5):
            labs = DBSCAN(eps=eps, min_samples=ms).fit_predict(X)
            if len(set(labs)) < 2: continue
            score = normalized_mutual_info_score(y_true, labs)
            if best is None or score > best[0]:
                best = (score, eps, ms, labs)
    if best is None:
        print(f'[{tag}/dbscan] no valid clustering found'); return
    score, eps, ms, labs = best
    print(f'[{tag}/dbscan] best eps={eps:.2f} min_samples={ms}')
    res_db = compare_clusters_to_truth(y_true, labs, tag=f'{tag}/dbscan')
    plot_confusion(res_db['aligned'], y_true, tag=f'{tag}/dbscan')
    return dict(kmeans=res_km, dbscan=res_db)

# run_cluster_eval(None, tag='coco')                              # baseline
# run_cluster_eval(RUNS_DIR / 'supcon_backbone.pt', tag='supcon') # after SupCon
# run_cluster_eval(RUNS_DIR / 'dino_backbone.pt',   tag='dino')   # after DINO

## Usage summary

Three files, three responsibilities:

| File                              | Responsibility                               |
| --------------------------------- | -------------------------------------------- |
| `augmentation_pipeline.ipynb`     | Build & inspect the augmentation pipeline    |
| `cnn_transfer.ipynb` (this file)  | Train and evaluate the model                 |
| `deploy.py`                       | Load weights, run on ZED / webcam / image    |

**Typical workflow**

1. Run `augmentation_pipeline.ipynb` end-to-end. Inspect Sections 6 and 6b.
   Tune `AUG_PROBS` if anything looks off.
2. Run this notebook. *Run All* now trains Section A, evaluates it, and
   writes weights to `runs_cnn_transfer/ft/weights/best.pt`.
3. Deploy from a terminal:
   ```bash
   # ZedBox
   python deploy.py --weights runs_cnn_transfer/ft/weights/best.pt --zed

   # laptop desk test
   python deploy.py --weights runs_cnn_transfer/ft/weights/best.pt --webcam 0

   # single-image smoke test
   python deploy.py --weights runs_cnn_transfer/ft/weights/best.pt \
                    --image data/raw/ambulance_0001.png
   ```

Sections B (SupCon) and C (DINO + cluster-vs-truth) remain available but
are kept with their invocations commented — they are opt-in research
paths, not part of the default training flow.

**Resuming a crashed run**

```python
resume_training(run_name='ft')   # from §A.4
```

**Further-training an existing `best.pt` on new data**

```python
DATA_YAML = build_split(IMAGES_DIR, LABELS_DIR, YOLO_DATASET_DIR)   # rebuild split
continue_training(
    weights   = RUNS_DIR / 'ft' / 'weights' / 'best.pt',
    data_yaml = DATA_YAML,
    epochs    = 35,
    run_name  = 'ft_glare_v1',
)
```
